In [160]:
"""
FlyWire 连接可视化

这个脚本使用多种方式展示 FlyWire 连接组的连接模式。
"""

'\nFlyWire 连接可视化\n\n这个脚本使用多种方式展示 FlyWire 连接组的连接模式。\n'

In [161]:
import sys
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # 使用非交互式后端

# 确保 UTF-8 编码
import locale
locale.setlocale(locale.LC_ALL, '')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict, Counter
import networkx as nx
from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist, squareform

In [ ]:
# 设置 UTF-8 和中文字体支持
import matplotlib.font_manager as fm

# 根据操作系统选择合适的字体
import platform
system = platform.system()

if system == 'Darwin':  # macOS
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti SC', 'STHeiti', 'DejaVu Sans']
elif system == 'Windows':
    plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'SimSun', 'DejaVu Sans']
else:  # Linux
    plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei', 'Noto Sans CJK SC', 'DejaVu Sans']

# 解决负号显示问题
plt.rcParams['axes.unicode_minus'] = False

# 设置默认编码
plt.rcParams['text.usetex'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

print(f"✓ 字体设置完成 (系统: {system})")
print(f"  当前字体: {plt.rcParams['font.sans-serif']}")

✓ 字体设置完成 (系统: Darwin)
  当前字体: ['Arial Unicode MS', 'PingFang SC', 'Heiti SC', 'STHeiti', 'DejaVu Sans']


In [163]:
import matplotlib.font_manager as fm
import platform
system = platform.system()

In [ ]:
if system == 'Darwin':  # macOS
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti SC', 'STHeiti', 'DejaVu Sans']
elif system == 'Windows':
    plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'SimSun', 'DejaVu Sans']
else:  # Linux
    plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei', 'Noto Sans CJK SC', 'DejaVu Sans']

In [ ]:
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['text.usetex'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
# 设置绘图样式
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    try:
        plt.style.use('seaborn-whitegrid')
    except:
        pass
sns.set_palette("husl")

In [164]:
# 设置输出目录
output_dir = Path('outputs/connections')
output_dir.mkdir(parents=True, exist_ok=True)

In [165]:
print("✓ 库导入成功")
print(f"✓ 输出目录: {output_dir}")

✓ 库导入成功
✓ 输出目录: outputs/connections


============================================================================
1. 加载数据
============================================================================

In [166]:
print("\n" + "="*60)
print("1. 加载 FlyWire 连接组数据")
print("="*60)


1. 加载 FlyWire 连接组数据


In [167]:
json_path = Path("../flyvis/connectome/flywire_v1.0.json")

In [168]:
with open(json_path, 'r') as f:
    flywire_data = json.load(f)

In [169]:
nodes = flywire_data['nodes']
edges = flywire_data['edges']
cell_types = [node['name'] for node in nodes]

In [170]:
print(f"✓ 加载完成")
print(f"  - 细胞类型: {len(cell_types)}")
print(f"  - 连接: {len(edges)}")

✓ 加载完成
  - 细胞类型: 146
  - 连接: 2071


In [171]:
# 构建连接矩阵
n_types = len(cell_types)
type_to_idx = {ct: i for i, ct in enumerate(cell_types)}
conn_matrix = np.zeros((n_types, n_types))
syn_matrix = np.zeros((n_types, n_types))

In [172]:
for edge in edges:
    src_idx = type_to_idx[edge['src']]
    tar_idx = type_to_idx[edge['tar']]
    conn_matrix[src_idx, tar_idx] = 1
    
    # 计算突触数
    if 'offsets' in edge and len(edge['offsets']) > 0:
        syn_count = sum(offset[1] for offset in edge['offsets'])
    else:
        syn_count = 1
    syn_matrix[src_idx, tar_idx] = syn_count

In [173]:
print(f"✓ 连接矩阵构建完成: {conn_matrix.shape}")

✓ 连接矩阵构建完成: (146, 146)


============================================================================
2. 网络图可视化
============================================================================

In [174]:
print("\n" + "="*60)
print("2. 网络图可视化")
print("="*60)


2. 网络图可视化


In [175]:
# 选择连接度最高的节点进行可视化
in_degree = defaultdict(int)
out_degree = defaultdict(int)
for edge in edges:
    out_degree[edge['src']] += 1
    in_degree[edge['tar']] += 1

In [176]:
total_degree = {ct: in_degree[ct] + out_degree[ct] for ct in cell_types}
top_nodes = sorted(total_degree.items(), key=lambda x: x[1], reverse=True)[:30]
top_node_names = [node[0] for node in top_nodes]

In [177]:
print(f"选择 Top 30 节点进行可视化")

选择 Top 30 节点进行可视化


In [178]:
# 构建 NetworkX 图
G = nx.DiGraph()
for edge in edges:
    if edge['src'] in top_node_names and edge['tar'] in top_node_names:
        if 'offsets' in edge and len(edge['offsets']) > 0:
            weight = sum(offset[1] for offset in edge['offsets'])
        else:
            weight = 1
        G.add_edge(edge['src'], edge['tar'], weight=weight)

In [179]:
print(f"✓ 网络图构建完成: {G.number_of_nodes()} 节点, {G.number_of_edges()} 边")

✓ 网络图构建完成: 30 节点, 360 边


In [180]:
# 绘制网络图
plt.figure(figsize=(16, 12))
pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

In [181]:
# 节点大小根据度数
node_sizes = [total_degree[node] * 50 for node in G.nodes()]

In [182]:
# 边宽度根据权重
edges_list = G.edges()
weights = [G[u][v]['weight'] for u, v in edges_list]
max_weight = max(weights) if weights else 1
edge_widths = [w / max_weight * 3 for w in weights]

In [183]:
# 绘制
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='lightblue', 
                       alpha=0.7, edgecolors='black', linewidths=1.5)
nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.5, 
                       edge_color='gray', arrows=True, arrowsize=15,
                       arrowstyle='->', connectionstyle='arc3,rad=0.1')
nx.draw_networkx_labels(G, pos, font_size=8, font_family='Arial Unicode MS')

{'Li33': Text(0.21574700874758151, 0.02548599236370543, 'Li33'),
 'LLPt': Text(0.1412037444253199, -0.5223063276679323, 'LLPt'),
 'Li30': Text(-0.08906784937632489, -0.25971473919728116, 'Li30'),
 'TmY31': Text(-0.03373685074653313, -0.2376854363531109, 'TmY31'),
 'Tm8a': Text(-0.1525352955138446, 1.2131194576560039e-05, 'Tm8a'),
 'Tm16': Text(-0.21460651736552014, -0.31956803350365615, 'Tm16'),
 'Mi4': Text(0.1651164219117011, 0.1260287733513207, 'Mi4'),
 'TmY11': Text(0.038182774697770164, -0.16357412921328077, 'TmY11'),
 'Tm5c': Text(-0.25550001225432234, 0.03266294446455733, 'Tm5c'),
 'Tm5e': Text(-0.1993166122314487, -0.08749412052817206, 'Tm5e'),
 'Tm5f': Text(0.34973643530685655, 0.380015682137275, 'Tm5f'),
 'Tm5b': Text(-0.08084618183836169, 0.44037950372902246, 'Tm5b'),
 'Li10': Text(-0.2912972399239366, -0.5597510818470567, 'Li10'),
 'Sm09': Text(-0.011732247526339801, 0.6643815142541009, 'Sm09'),
 'TmY10': Text(0.07708698747852143, -0.022249110273059813, 'TmY10'),
 'Tlp4': T

In [184]:
plt.title('FlyWire 连接网络图 (Top 30 节点)', fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.savefig(output_dir / 'network_graph.png', dpi=300, bbox_inches='tight')
plt.close()

In [185]:
print(f"✓ 网络图已保存")

✓ 网络图已保存


============================================================================
3. 连接矩阵热图
============================================================================

In [186]:
print("\n" + "="*60)
print("3. 连接矩阵热图")
print("="*60)


3. 连接矩阵热图


In [187]:
# 选择 Top 40 细胞类型
top_40_types = [node[0] for node in sorted(total_degree.items(), key=lambda x: x[1], reverse=True)[:40]]
top_40_indices = [type_to_idx[ct] for ct in top_40_types]

In [188]:
sub_matrix = syn_matrix[np.ix_(top_40_indices, top_40_indices)]

In [189]:
# 绘制热图
plt.figure(figsize=(14, 12))
sns.heatmap(sub_matrix, xticklabels=top_40_types, yticklabels=top_40_types,
            cmap='YlOrRd', cbar_kws={'label': '突触数量'}, linewidths=0.5)
plt.title('FlyWire 连接矩阵热图 (Top 40 细胞类型)', fontsize=14, fontweight='bold')
plt.xlabel('目标细胞类型', fontsize=12)
plt.ylabel('源细胞类型', fontsize=12)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(output_dir / 'connection_matrix_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()

In [190]:
print(f"✓ 连接矩阵热图已保存")

✓ 连接矩阵热图已保存


============================================================================
4. 层次聚类分析
============================================================================

In [191]:
print("\n" + "="*60)
print("4. 层次聚类分析")
print("="*60)


4. 层次聚类分析


In [192]:
# 基于连接模式进行聚类
features = np.hstack([sub_matrix, sub_matrix.T])  # 输入+输出连接
distances = pdist(features, metric='euclidean')
linkage_matrix = hierarchy.linkage(distances, method='ward')

In [193]:
# 绘制树状图
plt.figure(figsize=(16, 8))
dendro = hierarchy.dendrogram(linkage_matrix, labels=top_40_types, 
                              leaf_font_size=10, leaf_rotation=90)
plt.title('基于连接模式的层次聚类', fontsize=14, fontweight='bold')
plt.xlabel('细胞类型', fontsize=12)
plt.ylabel('距离', fontsize=12)
plt.tight_layout()
plt.savefig(output_dir / 'hierarchical_clustering.png', dpi=300, bbox_inches='tight')
plt.close()

In [194]:
print(f"✓ 层次聚类图已保存")

✓ 层次聚类图已保存


============================================================================
5. 输入-输出通路图
============================================================================

In [195]:
print("\n" + "="*60)
print("5. 输入-输出通路分析")
print("="*60)


5. 输入-输出通路分析


In [196]:
input_types = flywire_data['input_units']
output_types = flywire_data['output_units']

In [197]:
# 找到从输入到输出的所有中间神经元
intermediate_neurons = set()
for edge in edges:
    if edge['src'] in input_types:
        intermediate_neurons.add(edge['tar'])
    if edge['tar'] in output_types:
        intermediate_neurons.add(edge['src'])

In [198]:
# 移除输入和输出神经元
intermediate_neurons = intermediate_neurons - set(input_types) - set(output_types)

In [199]:
print(f"输入神经元: {len(input_types)}")
print(f"输出神经元: {len(output_types)}")
print(f"中间神经元: {len(intermediate_neurons)}")

输入神经元: 3
输出神经元: 8
中间神经元: 37


In [200]:
# 统计每个中间神经元的连接
intermediate_stats = []
for neuron in intermediate_neurons:
    from_input = sum(1 for e in edges if e['src'] in input_types and e['tar'] == neuron)
    to_output = sum(1 for e in edges if e['src'] == neuron and e['tar'] in output_types)
    if from_input > 0 or to_output > 0:
        intermediate_stats.append({
            'neuron': neuron,
            'from_input': from_input,
            'to_output': to_output,
            'total': from_input + to_output
        })

In [201]:
intermediate_stats = sorted(intermediate_stats, key=lambda x: x['total'], reverse=True)[:20]

In [202]:
# 绘制通路图
fig, ax = plt.subplots(figsize=(14, 10))

In [203]:
y_positions = {
    'input': 0.9,
    'intermediate': 0.5,
    'output': 0.1
}

In [204]:
# 绘制输入神经元
for i, neuron in enumerate(input_types):
    x = i / max(len(input_types) - 1, 1)
    ax.scatter(x, y_positions['input'], s=500, c='green', alpha=0.7, edgecolors='black', linewidths=2)
    ax.text(x, y_positions['input'] + 0.05, neuron, ha='center', fontsize=10, fontweight='bold')

In [205]:
# 绘制输出神经元
for i, neuron in enumerate(output_types):
    x = i / max(len(output_types) - 1, 1)
    ax.scatter(x, y_positions['output'], s=500, c='red', alpha=0.7, edgecolors='black', linewidths=2)
    ax.text(x, y_positions['output'] - 0.05, neuron, ha='center', va='top', fontsize=10, fontweight='bold')

In [206]:
# 绘制关键中间神经元
top_intermediate = [stat['neuron'] for stat in intermediate_stats[:10]]
for i, neuron in enumerate(top_intermediate):
    x = i / max(len(top_intermediate) - 1, 1)
    ax.scatter(x, y_positions['intermediate'], s=300, c='blue', alpha=0.6, edgecolors='black', linewidths=1.5)
    ax.text(x, y_positions['intermediate'], neuron, ha='center', va='center', fontsize=8)

In [207]:
ax.set_xlim(-0.1, 1.1)
ax.set_ylim(0, 1)
ax.set_title('输入-输出通路图 (Top 10 中间神经元)', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig(output_dir / 'input_output_pathway.png', dpi=300, bbox_inches='tight')
plt.close()

In [208]:
print(f"✓ 输入-输出通路图已保存")

✓ 输入-输出通路图已保存


============================================================================
6. 连接强度分布
============================================================================

In [209]:
print("\n" + "="*60)
print("6. 连接强度分布")
print("="*60)


6. 连接强度分布


In [210]:
# 统计突触数量分布
synapse_counts = []
for edge in edges:
    if 'offsets' in edge and len(edge['offsets']) > 0:
        syn_count = sum(offset[1] for offset in edge['offsets'])
        synapse_counts.append(syn_count)

In [211]:
if synapse_counts:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 直方图
    axes[0].hist(synapse_counts, bins=50, color='#4ecdc4', alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('突触数量', fontsize=12)
    axes[0].set_ylabel('连接数', fontsize=12)
    axes[0].set_title('突触数量分布', fontsize=14, fontweight='bold')
    axes[0].grid(alpha=0.3)
    
    # 对数尺度
    axes[1].hist(synapse_counts, bins=50, color='#ff6b6b', alpha=0.7, edgecolor='black')
    axes[1].set_xlabel('突触数量', fontsize=12)
    axes[1].set_ylabel('连接数', fontsize=12)
    axes[1].set_yscale('log')
    axes[1].set_title('突触数量分布 (对数尺度)', fontsize=14, fontweight='bold')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_dir / 'synapse_distribution.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✓ 连接强度分布图已保存")
    print(f"  - 平均突触数: {np.mean(synapse_counts):.1f}")
    print(f"  - 中位数: {np.median(synapse_counts):.1f}")
    print(f"  - 最大值: {np.max(synapse_counts)}")
    print(f"  - 最小值: {np.min(synapse_counts)}")

✓ 连接强度分布图已保存
  - 平均突触数: 990.5
  - 中位数: 92.0
  - 最大值: 90970
  - 最小值: 10


============================================================================
7. 总结
============================================================================

In [212]:
print("\n" + "="*60)
print("FlyWire 连接可视化总结")
print("="*60)
print(f"\n✓ 生成的可视化:")
print(f"  1. 网络图 (Top 30 节点)")
print(f"  2. 连接矩阵热图 (Top 40 细胞类型)")
print(f"  3. 层次聚类树状图")
print(f"  4. 输入-输出通路图")
print(f"  5. 连接强度分布")
print(f"\n✓ 所有图表已保存到: {output_dir}")
print(f"\n🎉 可视化完成！")
print("="*60)


FlyWire 连接可视化总结

✓ 生成的可视化:
  1. 网络图 (Top 30 节点)
  2. 连接矩阵热图 (Top 40 细胞类型)
  3. 层次聚类树状图
  4. 输入-输出通路图
  5. 连接强度分布

✓ 所有图表已保存到: outputs/connections

🎉 可视化完成！
